In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery
from google.cloud.exceptions import NotFound


In [2]:
# 1. Carregar credenciais
load_dotenv()
client = bigquery.Client()

In [3]:
# 1. O ID do Projeto (A Nuvem)
# DEVE ser o ID real que está no seu console do Google Cloud (ou pego via client.project)
PROJECT_ID = client.project
# 2. O seu "Banco de Dados" (Dataset)
# Aqui sim entra o nome que você escolheu!
DB_NAME = "analytics_ford"
# 3. A sua Tabela
TABLE_NAME = "marcas_veiculos"

# Formato: projeto.dataset
DATASET_ID = f"{PROJECT_ID}.{DB_NAME}" 

# Formato: projeto.dataset.tabela
TABLE_ID = f"{DATASET_ID}.{TABLE_NAME}" 

print(f"O caminho final da tabela será: {TABLE_ID}")
# Resultado esperado: id-do-seu-projeto-gcp.analytics_ford.marcas_veiculos

O caminho final da tabela será: ford-493401.analytics_ford.marcas_veiculos


In [ ]:
def preparar_infraestrutura():
    """Cria o dataset e a tabela caso não existam."""
    
    # Criar Dataset se não existir
    try:
        client.get_dataset(DATASET_ID)
        print(f"Dataset {DATASET_ID} já existe.")
    except NotFound:
        dataset = bigquery.Dataset(DATASET_ID)
        dataset.location = "US"  # Escolha a região
        client.create_dataset(dataset)
        print(f"Dataset {DATASET_ID} criado.")

    # Definir o Schema (Estrutura da Tabela)
    schema = [
        bigquery.SchemaField("tipo_veiculo", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("marca_nome", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("marca_codigo", "STRING", mode="REQUIRED"),
    ]

    # Criar Tabela se não existir
    try:
        client.get_table(TABLE_ID)
        print(f"Tabela {TABLE_ID} já existe.")
    except NotFound:
        table = bigquery.Table(TABLE_ID, schema=schema)
        client.create_table(table)
        print(f"Tabela {TABLE_ID} criada.")

def salvar_no_bigquery(df):
    """Ingere o DataFrame do Pandas na tabela do BigQuery."""
    print(f"Iniciando ingestão de {len(df)} linhas no BigQuery...")
    
    # Configuração da carga: sobrescrever dados antigos (WRITE_TRUNCATE) 
    # ou apenas adicionar (WRITE_APPEND)
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE", 
    )

    job = client.load_table_from_dataframe(df, TABLE_ID, job_config=job_config)
    job.result()  # Aguarda a conclusão
    
    print(f"Dados salvos com sucesso na tabela {TABLE_ID}!")

# --- EXECUÇÃO ---
if __name__ == "__main__":
    # 1. Garantir que o banco e a tabela existem
    preparar_infraestrutura()
    
    # 2. Chamar sua função de extração (que você postou acima)
    # Supondo que a função extrair_mapeamento_marcas() esteja no mesmo arquivo
    # df_resultado = extrair_mapeamento_marcas()
    
    # 3. Salvar o resultado
    # salvar_no_bigquery(df_resultado)

Dataset ford-493401.analytics_ford já existe.
Tabela ford-493401.analytics_ford.marcas_veiculos criada.


: 

In [18]:
query = f"""
    SELECT 
        tipo_veiculo, 
        marca_nome, 
        marca_codigo
    FROM 
        `{PROJECT_ID}.analytics_ford.marcas_veiculos`
    WHERE tipo_veiculo IN ('carros')
    ORDER BY 
        tipo_veiculo, marca_nome

"""
    
# 3. Executar a consulta (Job)
# query_job = client.query(query)
    
  
df_resultado = client.query(query).to_dataframe()
    


df_resultado

c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,tipo_veiculo,marca_nome,marca_codigo
0,carros,AM Gen,4
1,carros,ASTON MARTIN,189
2,carros,Acura,1
3,carros,Agrale,2
4,carros,Alfa Romeo,3
...,...,...,...
101,carros,Volvo,58
102,carros,Wake,163
103,carros,Walk,120
104,carros,ZEEKR,253


In [20]:
df_resultado

,tipo_veiculo,marca_nome,marca_codigo
0,carros,AM Gen,4
1,carros,ASTON MARTIN,189
2,carros,Acura,1
3,carros,Agrale,2
4,carros,Alfa Romeo,3
...,...,...,...
101,carros,Volvo,58
102,carros,Wake,163
103,carros,Walk,120
104,carros,ZEEKR,253


In [29]:
df_resultado['marca_codigo'] = df_resultado['marca_codigo'].astype(str).str.lower()

In [32]:
df_resultado['marca_nome'] = df_resultado['marca_nome'].astype(str).str.lower()

In [36]:
codigo_ford = df_resultado[df_resultado['marca_nome'] == 'ford']['marca_codigo'].iloc[0]

In [37]:
codigo_ford

'22'

In [38]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

# ==========================================
# 1. CONFIGURAÇÃO E CREDENCIAIS
# ==========================================
load_dotenv()
client = bigquery.Client()
project_id = client.project

DB_NAME = "analytics_ford"
TABELA_ORIGEM = f"{project_id}.{DB_NAME}.marcas_veiculos"
TABELA_DESTINO = f"{project_id}.{DB_NAME}.modelos_veiculos"

# ==========================================
# 2. FUNÇÕES DE INFRAESTRUTURA E API
# ==========================================
def preparar_tabela_modelos():
    """Cria a tabela de destino para os modelos, se não existir."""
    schema = [
        bigquery.SchemaField("tipo_veiculo", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("marca_codigo", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("modelo_codigo", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("modelo_nome", "STRING", mode="REQUIRED"),
    ]
    try:
        client.get_table(TABELA_DESTINO)
    except NotFound:
        table = bigquery.Table(TABELA_DESTINO, schema=schema)
        client.create_table(table)
        print(f"✨ Tabela {TABELA_DESTINO} criada.")

def extrair_modelos_por_marca(cod_marca, tipo_veiculo):
    """Consulta a API da Fipe para extrair modelos de uma marca específica."""
    base_url = "https://parallelum.com.br/fipe/api/v1"
    headers = {'User-Agent': 'Mozilla/5.0'}
    url = f"{base_url}/{tipo_veiculo}/marcas/{cod_marca}/modelos"
    
    print(f"Buscando na API: {tipo_veiculo.upper()} | ID: {cod_marca}...")

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        lista_modelos = response.json().get('modelos', [])
        if not lista_modelos: return pd.DataFrame()

        df_modelos = pd.DataFrame(lista_modelos)
        df_modelos = df_modelos.rename(columns={'codigo': 'modelo_codigo', 'nome': 'modelo_nome'})
        
        df_modelos['marca_codigo'] = str(cod_marca)
        df_modelos['tipo_veiculo'] = str(tipo_veiculo)
        df_modelos['modelo_codigo'] = df_modelos['modelo_codigo'].astype(str)
        df_modelos['modelo_nome'] = df_modelos['modelo_nome'].astype(str)

        return df_modelos[['tipo_veiculo', 'marca_codigo', 'modelo_codigo', 'modelo_nome']]

    except Exception as e:
        print(f"❌ Erro na API para a marca {cod_marca}: {e}")
        return pd.DataFrame()

# ==========================================
# 3. NOVO PIPELINE SOB DEMANDA
# ==========================================
def executar_pipeline_por_marca(nome_marca_alvo):
    print("--- 1. Preparando Infraestrutura ---")
    preparar_tabela_modelos()
    
    print(f"\n--- 2. Buscando código da marca: '{nome_marca_alvo}' no BigQuery ---")
    # Usamos LOWER para ignorar se você digitou 'Ford', 'FORD' ou 'ford'
    query = f"""
        SELECT tipo_veiculo, marca_codigo, marca_nome 
        FROM `{TABELA_ORIGEM}`
        WHERE LOWER(marca_nome) = LOWER('{nome_marca_alvo}')
    """
    
    df_marcas = client.query(query).to_dataframe()
    
    if df_marcas.empty:
        print(f"❌ A marca '{nome_marca_alvo}' não foi encontrada na tabela {TABELA_ORIGEM}.")
        return

    print(f"✅ Encontrada(s) {len(df_marcas)} correspondência(s) para '{nome_marca_alvo}':")
    for _, row in df_marcas.iterrows():
        print(f"   -> Categoria: {row['tipo_veiculo']} | Código: {row['marca_codigo']}")

    print("\n--- 3. Extraindo Modelos da API ---")
    lista_dfs = []
    
    for index, row in df_marcas.iterrows():
        df_resultado = extrair_modelos_por_marca(row['marca_codigo'], row['tipo_veiculo'])
        if not df_resultado.empty:
            lista_dfs.append(df_resultado)
        time.sleep(0.5) 
        
    print("\n--- 4. Salvando no BigQuery ---")
    if lista_dfs:
        df_final = pd.concat(lista_dfs, ignore_index=True)
        print(f"Total de modelos extraídos: {len(df_final)}. Iniciando upload...")
        
        # MUDANÇA IMPORTANTE: WRITE_APPEND adiciona sem apagar o resto!
        job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
        job = client.load_table_from_dataframe(df_final, TABELA_DESTINO, job_config=job_config)
        job.result()
        
        print(f"🎉 Sucesso! Modelos da '{nome_marca_alvo}' adicionados à tabela {TABELA_DESTINO}.")
    else:
        print("Nenhum modelo foi extraído.")

# ==========================================
# EXECUÇÃO
# ==========================================
if __name__ == "__main__":
    # Teste digitando o nome da marca que você quer extrair!
    # Pode ser "Ford", "Fiat", "Honda", "Yamaha"...
    marca_desejada = input("Digite o nome da marca que deseja extrair: ")
    
    executar_pipeline_por_marca(marca_desejada)

--- 1. Preparando Infraestrutura ---
✨ Tabela ford-493401.analytics_ford.modelos_veiculos criada.

--- 2. Buscando código da marca: 'ford' no BigQuery ---


c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Encontrada(s) 2 correspondência(s) para 'ford':
   -> Categoria: caminhoes | Código: 105
   -> Categoria: carros | Código: 22

--- 3. Extraindo Modelos da API ---
Buscando na API: CAMINHOES | ID: 105...
Buscando na API: CARROS | ID: 22...

--- 4. Salvando no BigQuery ---
Total de modelos extraídos: 651. Iniciando upload...


c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


🎉 Sucesso! Modelos da 'ford' adicionados à tabela ford-493401.analytics_ford.modelos_veiculos.


In [ ]:
query = f"""
    SELECT 
      *
    FROM 
        `{PROJECT_ID}.analytics_ford.modelos_veiculos`


"""
    
# 3. Executar a consulta (Job)
# query_job = client.query(query)
    
  
df_resultado = client.query(query).to_dataframe()
    


df_resultado

c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,tipo_veiculo,marca_codigo,modelo_codigo,modelo_nome
0,caminhoes,105,3151,CARGO 1113 2p (diesel)
1,caminhoes,105,3152,CARGO 1113 3-Eixos 2p (diesel)
2,caminhoes,105,3153,CARGO 1114 2p (diesel)
3,caminhoes,105,3154,CARGO 1114 3-Eixos 2p (diesel)
4,caminhoes,105,3155,CARGO 1117 T 3-Eixos 2p (diesel)
...,...,...,...,...
646,carros,22,921,Versailles Ghia 2.0i / 2.0 2p e 4p
647,carros,22,922,Versailles GL 1.8i / 1.8 2p e 4p
648,carros,22,923,Versailles GL 2.0i / 2.0 2p e 4p
649,carros,22,924,Windstar GL


In [42]:
df_resultado_carros = df_resultado[df_resultado['tipo_veiculo'] == 'carros']

In [45]:
df_resultado_carros.query("modelo_nome.str.lower().str.contains('territory')")

,tipo_veiculo,marca_codigo,modelo_codigo,modelo_nome
614,carros,22,9163,TERRITORY SEL 1.5 GTDi EcoBoost Aut.
615,carros,22,9164,TERRITORY Titanium 1.5 GTDi EcoBo. Aut.


In [46]:
tipo_veiculo = 'carros'
cod_marca = '22'  # Exemplo: código da marca Ford para carros
base_url = "https://parallelum.com.br/fipe/api/v1"
headers = {'User-Agent': 'Mozilla/5.0'}
url = f"{base_url}/{tipo_veiculo}/marcas/{cod_marca}/modelos"

In [48]:
url_anos = f"{base_url}/{tipo_veiculo}/marcas/{cod_marca}/modelos/{9164}/anos"

In [49]:
url_anos

'https://parallelum.com.br/fipe/api/v1/carros/marcas/22/modelos/9164/anos'

In [47]:
url

'https://parallelum.com.br/fipe/api/v1/carros/marcas/22/modelos'

In [51]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

# ==========================================
# 1. CONFIGURAÇÃO
# ==========================================
load_dotenv()
client = bigquery.Client()
project_id = client.project or os.getenv("GOOGLE_CLOUD_PROJECT")

# Usando exatamente as tabelas que você definiu
DB_NAME = "analytics_ford"
TABELA_ORIGEM = f"{project_id}.{DB_NAME}.modelos_veiculos"
TABELA_DESTINO = f"{project_id}.{DB_NAME}.modelos_com_fipe"

TAMANHO_DO_LOTE = 50  # Salva no BigQuery a cada 50 modelos encontrados

# ==========================================
# 2. FUNÇÕES BASE
# ==========================================
def preparar_tabela_destino():
    """Garante que a tabela final exista antes de começar."""
    schema = [
        bigquery.SchemaField("tipo_veiculo", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("marca_codigo", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("modelo_codigo", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("modelo_nome", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("codigo_fipe", "STRING", mode="REQUIRED"),
    ]
    try:
        client.get_table(TABELA_DESTINO)
    except NotFound:
        table = bigquery.Table(TABELA_DESTINO, schema=schema)
        client.create_table(table)
        print(f"✨ Tabela de destino {TABELA_DESTINO} criada.")

def buscar_fipe_na_api(cod_marca, cod_modelo, tipo_veiculo):
    """Bate na API da Fipe e retorna o código oficial usando Token de Autenticação."""
    base_url = "https://parallelum.com.br/fipe/api/v1"
    
    # 1. Puxa o token do arquivo .env
    token_fipe = os.getenv("FIPE_API_TOKEN")
    
    # 2. Adiciona o token no cabeçalho da requisição
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'X-Subscription-Token': token_fipe
    }
    
    url_anos = f"{base_url}/{tipo_veiculo}/marcas/{cod_marca}/modelos/{cod_modelo}/anos"

    try:
        resp_anos = requests.get(url_anos, headers=headers)
        resp_anos.raise_for_status()
        anos = resp_anos.json()

        if not anos: return None

        primeiro_ano = anos[0]['codigo']
        url_final = f"{base_url}/{tipo_veiculo}/marcas/{cod_marca}/modelos/{cod_modelo}/anos/{primeiro_ano}"
        
        # A segunda requisição também usa o mesmo 'headers' com o token
        resp_final = requests.get(url_final, headers=headers)
        resp_final.raise_for_status()

        return resp_final.json().get('CodigoFipe')

    except Exception as e:
        # Opcional: imprimir o erro para debug caso o token seja recusado (Erro 401 ou 403)
        print(f"Erro na API: {e}")
        return None

# ==========================================
# 3. MOTOR PRINCIPAL (PROCESSAMENTO EM LOTES)
# ==========================================
def rodar_extracao_massiva():
    preparar_tabela_destino()
    
    print("\n🔍 Analisando o banco de dados para descobrir o que falta...")
    
    # Esta query é mágica: Ela pega todos os modelos, cruza com a tabela de destino
    # e só retorna quem AINDA NÃO TEM Fipe salvo.
    query = f"""
        SELECT 
            o.tipo_veiculo, o.marca_codigo, o.modelo_codigo, o.modelo_nome
        FROM 
            `{TABELA_ORIGEM}` o
        LEFT JOIN 
            `{TABELA_DESTINO}` d
          ON o.modelo_codigo = d.modelo_codigo AND o.marca_codigo = d.marca_codigo
        WHERE 
            d.codigo_fipe IS NULL
    """
    
    df_pendentes = client.query(query).to_dataframe()
    total = len(df_pendentes)
    
    if total == 0:
        print("✅ Parabéns! Todos os modelos do banco já possuem código FIPE.")
        return

    print(f"🚀 Iniciando extração de {total} modelos pendentes. Lotes de {TAMANHO_DO_LOTE}.")
    print("-" * 50)

    lote_atual = []
    
    # Iterar sobre as linhas pendentes
    for index, row in df_pendentes.iterrows():
        cod_marca = row['marca_codigo']
        cod_modelo = row['modelo_codigo']
        tipo = row['tipo_veiculo']
        nome = row['modelo_nome']
        
        print(f"[{index + 1}/{total}] Buscando: {nome[:25].ljust(25)}...", end=" ")
        
        fipe = buscar_fipe_na_api(cod_marca, cod_modelo, tipo)
        
        if fipe:
            print(f"OK ({fipe})")
            # Adiciona ao lote em formato de dicionário
            lote_atual.append({
                "tipo_veiculo": tipo,
                "marca_codigo": cod_marca,
                "modelo_codigo": cod_modelo,
                "modelo_nome": nome,
                "codigo_fipe": fipe
            })
        else:
            print("FALHOU")
            
        # Pausa obrigatória para a API não bloquear seu IP
        time.sleep(0.5)
        
        # --- LÓGICA DE SALVAMENTO EM LOTE ---
        if len(lote_atual) >= TAMANHO_DO_LOTE:
            print(f"\n💾 Salvando lote de {len(lote_atual)} registros no BigQuery...")
            df_lote = pd.DataFrame(lote_atual)
            
            job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
            job = client.load_table_from_dataframe(df_lote, TABELA_DESTINO, job_config=job_config)
            job.result()
            
            print("✔️ Lote salvo com sucesso!\n")
            lote_atual = [] # Limpa o lote para começar o próximo

    # --- SALVAR O RESTO (Se sobrou algo no último lote) ---
    if len(lote_atual) > 0:
        print(f"\n💾 Salvando lote final de {len(lote_atual)} registros no BigQuery...")
        df_lote = pd.DataFrame(lote_atual)
        job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
        client.load_table_from_dataframe(df_lote, TABELA_DESTINO, job_config=job_config).result()
        print("✔️ Último lote salvo com sucesso!")

if __name__ == "__main__":
    rodar_extracao_massiva()


🔍 Analisando o banco de dados para descobrir o que falta...


c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


🚀 Iniciando extração de 410 modelos pendentes. Lotes de 50.
--------------------------------------------------
[1/410] Buscando: Escort RS 1.8i 16V       ... OK (003095-3)
[2/410] Buscando: Escort S.W GL 1.8i 16V   ... OK (003006-6)
[3/410] Buscando: Escort S.W. GLX 1.8i 16V ... OK (003008-2)
[4/410] Buscando: Escort SW 1.9 16v        ... OK (003230-1)
[5/410] Buscando: Escort SW GL 1.6 MPI     ... OK (003208-5)
[6/410] Buscando: Escort XR3 1.8 / 1.6 Bene... OK (003038-4)
[7/410] Buscando: Escort XR3 1.8 / 1.6 Conv... OK (003039-2)
[8/410] Buscando: Escort XR3 2.0i          ... OK (003040-6)
[9/410] Buscando: Escort XR3 2.0i Conversív... OK (003041-4)
[10/410] Buscando: E-TRANSIT Furgão Longo Te... OK (003514-9)
[11/410] Buscando: Expedition 5.4 V8        ... OK (003154-2)
[12/410] Buscando: Explorer Limited 4.0 4x4 ... OK (003290-5)
[13/410] Buscando: Explorer Limited 5.0 4x4 ... OK (003138-0)
[14/410] Buscando: Explorer Sport 4.0 V6    ... OK (003157-7)
[15/410] Buscando: Explorer XL

c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✔️ Lote salvo com sucesso!

[51/410] Buscando: F-1000 XLT/XL Lighting 4.... OK (003011-2)
[52/410] Buscando: F-150 Lariat 4x4 5.0 V8  ... OK (003489-4)
[53/410] Buscando: F-150 Lariat Black 4x4 5.... OK (003509-2)
[54/410] Buscando: F-150 Platinum 4x4 5.0 V8... OK (003490-8)
[55/410] Buscando: F-150 Tremor 4x4 5.0 V8  ... OK (003512-2)
[56/410] Buscando: F-150 XL Triton 4.3 V6   ... OK (003126-7)
[57/410] Buscando: F-150 XLT Triton 4.3     ... OK (003159-3)
[58/410] Buscando: F-150 XLT Triton 4.6 V8  ... OK (003217-4)
[59/410] Buscando: F-150 XLT Triton 5.8     ... OK (003160-7)
[60/410] Buscando: F-250 F-MILHA CD 3.9 TB D... OK (003338-3)
[61/410] Buscando: F-250 Tropica./Tropmine E... OK (003355-3)
[62/410] Buscando: F-250 TropiCab CE 3.9 TB ... OK (003313-8)
[63/410] Buscando: F-250 Tropical 3.9 Diesel... OK (003170-4)
[64/410] Buscando: F-250 Tropical 4.2 CE / C... OK (003172-0)
[65/410] Buscando: F-250 Tropical 4.2 V6    ... OK (003171-2)
[66/410] Buscando: F-250 Tropical SL/ Van 

...

✔️ Lote salvo com sucesso!

[351/410] Buscando: Ranger XLT 2.8 8v 135cv 4... OK (003253-0)
[352/410] Buscando: Ranger XLT 2.8 8v 135cv 4... OK (003251-4)
[353/410] Buscando: Ranger XLT 3.0 PSE 163cv ... OK (003294-8)
[354/410] Buscando: Ranger XLT 3.0 PSE 163cv ... OK (003295-6)
[355/410] Buscando: Ranger XLT 3.0 V6 4x4 CD ... OK (003496-7)
[356/410] Buscando: Ranger XLT 3.2 20V 4x4 CD... OK (003362-6)
[357/410] Buscando: Ranger XLT 3.2 20V 4x4 CD... OK (003363-4)
[358/410] Buscando: Ranger XLT 4.0 12v V6 210... OK (003240-9)
[359/410] Buscando: Ranger XLT 4.0 12v V6 210... OK (003239-5)
[360/410] Buscando: Ranger XLT 4.0 4x2 CE    ... OK (003149-6)
[361/410] Buscando: Ranger XLT 4.0 4x2 CS    ... OK (003088-0)
[362/410] Buscando: Ranger XLT 4.0 4x4 CD    ... OK (003112-7)
[363/410] Buscando: Ranger XLT 4.0 4x4 CE    ... OK (003150-0)
[364/410] Buscando: Ranger XLT 4.0 4x4 CS    ... OK (003111-9)
[365/410] Buscando: Ranger XLT Limit./L.Cent.... OK (003263-8)
[366/410] Buscando: Royale 

c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✔️ Lote salvo com sucesso!

[401/410] Buscando: Verona GLX 1.8i / 1.8 4p ... OK (003063-5)
[402/410] Buscando: Verona GLX 2.0i / 2.0 4p ... OK (003064-3)
[403/410] Buscando: Verona LX 1.6 (Modelo ant... OK (003067-8)
[404/410] Buscando: Verona LX 1.8 (Modelo ant... OK (003068-6)
[405/410] Buscando: Verona S 2.0i 4p         ... OK (003066-0)
[406/410] Buscando: Versailles Ghia 2.0i / 2.... OK (003072-4)
[407/410] Buscando: Versailles GL 1.8i / 1.8 ... OK (003070-8)
[408/410] Buscando: Versailles GL 2.0i / 2.0 ... OK (003071-6)
[409/410] Buscando: Windstar GL              ... OK (003093-7)
[410/410] Buscando: Windstar LX              ... OK (003094-5)

💾 Salvando lote final de 10 registros no BigQuery...


c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✔️ Último lote salvo com sucesso!


In [52]:
query = f"""
    SELECT *
    FROM 
        `{PROJECT_ID}.analytics_ford.modelos_com_fipe`

"""
    
# 3. Executar a consulta (Job)
# query_job = client.query(query)
    
  
df_resultado = client.query(query).to_dataframe()
    


df_resultado

c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,tipo_veiculo,marca_codigo,modelo_codigo,modelo_nome,codigo_fipe
0,carros,22,916,Verona GLX 1.8i / 1.8 4p,003063-5
1,carros,22,917,Verona GLX 2.0i / 2.0 4p,003064-3
2,carros,22,918,Verona LX 1.6 (Modelo antigo),003067-8
3,carros,22,919,Verona LX 1.8 (Modelo antigo),003068-6
4,carros,22,920,Verona S 2.0i 4p,003066-0
...,...,...,...,...,...
646,carros,22,887,Ranger XLT 2.5 4x2 CS Diesel,003203-4
647,carros,22,888,Ranger XLT 2.5 4x4 CD Diesel,003110-0
648,carros,22,889,Ranger XLT 2.5 4x4 CE TB Diesel,003148-8
649,carros,22,890,Ranger XLT 2.5 4x4 CS TB Diesel,003109-7


In [53]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

# ==========================================
# 1. CONFIGURAÇÃO
# ==========================================
load_dotenv()
client = bigquery.Client()

project_id = client.project 

DB_NAME = "analytics_ford"
TABELA_DESTINO = f"{project_id}.{DB_NAME}.referencia_meses_fipe"

# Dicionário para "traduzir" o mês em texto para número e abreviação
MAPEAMENTO_MESES = {
    'janeiro':   ('01', 'jan'),
    'fevereiro': ('02', 'fev'),
    'março':     ('03', 'mar'),
    'abril':     ('04', 'abr'),
    'maio':      ('05', 'mai'),
    'junho':     ('06', 'jun'),
    'julho':     ('07', 'jul'),
    'agosto':    ('08', 'ago'),
    'setembro':  ('09', 'set'),
    'outubro':   ('10', 'out'),
    'novembro':  ('11', 'nov'),
    'dezembro':  ('12', 'dez')
}

# ==========================================
# 2. PREPARAR INFRAESTRUTURA
# ==========================================
def preparar_tabela_referencia():
    """Garante que a tabela de referência exista."""
    schema = [
        bigquery.SchemaField("codigo", "INT64", mode="REQUIRED", description="Código oficial da Tabela Fipe"),
        bigquery.SchemaField("descricao_original", "STRING", mode="REQUIRED", description="Ex: abril/2026"),
        bigquery.SchemaField("ano", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("mes", "STRING", mode="REQUIRED", description="Mês em formato numérico (01 a 12)"),
        bigquery.SchemaField("mes_abrev", "STRING", mode="REQUIRED", description="Abreviação do mês (jan, fev...)"),
    ]
    try:
        client.get_table(TABELA_DESTINO)
        print(f"✅ Tabela {TABELA_DESTINO} já existe.")
    except NotFound:
        table = bigquery.Table(TABELA_DESTINO, schema=schema)
        client.create_table(table)
        print(f"✨ Tabela {TABELA_DESTINO} criada com sucesso.")

# ==========================================
# 3. EXTRAÇÃO E TRANSFORMAÇÃO (PANDAS)
# ==========================================
def extrair_e_transformar_referencias():
    url = "https://brasilapi.com.br/api/fipe/tabelas/v1"
    print("Buscando dados na BrasilAPI...")
    
    response = requests.get(url)
    response.raise_for_status()
    
    # Transforma o JSON direto em DataFrame
    df = pd.DataFrame(response.json())
    
    print("Limpando e transformando os dados...")
    
    # 1. Limpa espaços invisíveis que possam vir da API (ex: "abril/2026 ")
    df['descricao_original'] = df['mes'].str.strip()
    
    # 2. Quebra a coluna no símbolo '/' em duas novas colunas (nome_mes e ano_texto)
    df[['nome_mes', 'ano_texto']] = df['descricao_original'].str.lower().str.split('/', expand=True)
    
    # 3. Mapeia o nome do mês para pegar a tupla (numero, abreviação)
    # Se por acaso vier um mês que não conhecemos, coloca nulo
    df['dados_mes'] = df['nome_mes'].map(MAPEAMENTO_MESES)
    
    # 4. Extrai os dados da tupla para colunas separadas
    df['mes'] = df['dados_mes'].apply(lambda x: x[0] if pd.notnull(x) else '00')
    df['mes_abrev'] = df['dados_mes'].apply(lambda x: x[1] if pd.notnull(x) else 'unk')
    
    # 5. Converte os tipos (Ano e Código são inteiros)
    df['ano'] = df['ano_texto'].astype(int)
    df['codigo'] = df['codigo'].astype(int)
    
    # 6. Filtra apenas as colunas que vamos salvar e na ordem certa
    df_final = df[['codigo', 'descricao_original', 'ano', 'mes', 'mes_abrev']]
    
    return df_final

# ==========================================
# 4. EXECUÇÃO PRINCIPAL
# ==========================================
def executar():
    preparar_tabela_referencia()
    
    df_resultado = extrair_e_transformar_referencias()
    print(f"\nAmostra dos dados transformados ({len(df_resultado)} linhas no total):")
    print(df_resultado.head())
    
    print("\nSalvando no BigQuery...")
    # ATENÇÃO: Como esta é uma tabela de referência oficial que atualiza todo mês,
    # usamos WRITE_TRUNCATE. Isso apaga a antiga e salva a nova zerada.
    # Assim você nunca tem códigos duplicados!
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    
    job = client.load_table_from_dataframe(df_resultado, TABELA_DESTINO, job_config=job_config)
    job.result()
    
    print(f"🎉 Sucesso! Tabela de referência atualizada.")

if __name__ == "__main__":
    executar()

✨ Tabela ford-493401.analytics_ford.referencia_meses_fipe criada com sucesso.
Buscando dados na BrasilAPI...
Limpando e transformando os dados...

Amostra dos dados transformados (304 linhas no total):
   codigo descricao_original   ano mes mes_abrev
0     332         abril/2026  2026  04       abr
1     331         março/2026  2026  03       mar
2     330     fevereiro/2026  2026  02       fev
3     329       janeiro/2026  2026  01       jan
4     328      dezembro/2025  2025  12       dez

Salvando no BigQuery...


c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


🎉 Sucesso! Tabela de referência atualizada.


In [ ]:
query = f"""
    SELECT *
    FROM 
        `{PROJECT_ID}.analytics_ford.referencia_meses_fipe`

"""
    
# 3. Executar a consulta (Job)
# query_job = client.query(query)
    
  
df_resultado = client.query(query).to_dataframe()
    


df_resultado

c:\Users\guilh\Documents\Project\Ford\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,codigo,descricao_original,ano,mes,mes_abrev
0,332,abril/2026,2026,04,abr
1,320,abril/2025,2025,04,abr
2,308,abril/2024,2024,04,abr
3,296,abril/2023,2023,04,abr
4,284,abril/2022,2022,04,abr
...,...,...,...,...,...
299,55,setembro/2009,2009,09,set
300,43,setembro/2008,2008,09,set
301,30,setembro/2007,2007,09,set
302,16,setembro/2006,2006,09,set


: 